# LeetCode 1604 — Alert Using Same Key-Card Three or More Times in a One Hour Period bằng Python

Link đề: https://leetcode.com/problems/alert-using-same-key-card-three-or-more-times-in-a-one-hour-period/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def alertNames(self, keyName: List[str], keyTime: List[str]) -> List[str]:
        ...
```

## 1. Mục tiêu học

1. Chuẩn hóa dữ liệu thời gian: đổi `"HH:MM"` thành **số phút từ 0:00** để so sánh và trừ được.
2. **Gom nhóm theo khóa** (tên nhân viên) bằng `defaultdict(list)`, rồi **sắp xếp** từng nhóm.
3. Nhận ra mẹo **cửa sổ 3 phần tử liên tiếp**: trong dãy đã sắp xếp, có 3 lần quẹt trong 60 phút ⇔ tồn tại `i` với `t[i] − t[i−2] ≤ 60`.
4. Mở rộng sang **cửa sổ trượt hai con trỏ** cho bài tổng quát "`k` lần trong `w` phút".
5. Chú ý chi tiết đề: biên **bao gồm** (`10:00`–`11:00` tính là trong 1 giờ), kết quả **không trùng** và **sắp xếp theo tên**.

## 2. Đề bài & ví dụ

Nhân viên công ty dùng thẻ từ để mở cửa. Mỗi lần quẹt, hệ thống lưu **tên** và **thời điểm**. Hệ thống phát **cảnh báo** nếu một nhân viên quẹt thẻ **từ 3 lần trở lên trong một khoảng một giờ**.

Cho hai danh sách `keyName` và `keyTime`, trong đó `[keyName[i], keyTime[i]]` là tên người và thời điểm quẹt thẻ trong **cùng một ngày**. Thời gian theo định dạng 24 giờ `"HH:MM"` (ví dụ `"23:51"`, `"09:49"`).

Trả về danh sách **tên không trùng** của những người bị cảnh báo, **sắp xếp tăng dần theo bảng chữ cái**.

Lưu ý: `"10:00"` – `"11:00"` được coi là **trong** một giờ, còn `"22:51"` – `"23:52"` thì **không**.

**Ví dụ 1:** `keyName = ["daniel","daniel","daniel","luis","luis","luis","luis"]`, `keyTime = ["10:00","10:40","11:00","09:00","11:00","13:00","15:00"]` → `["daniel"]`.

**Ví dụ 2:** `keyName = ["alice","alice","alice","bob","bob","bob","bob"]`, `keyTime = ["12:01","12:00","18:00","21:00","21:20","21:30","23:00"]` → `["bob"]`.

**Ràng buộc:**
- `1 ≤ keyName.length, keyTime.length ≤ 10⁵`, `keyName.length == keyTime.length`
- `keyTime[i]` có dạng `"HH:MM"`.
- Mỗi cặp `[keyName[i], keyTime[i]]` là **duy nhất**.
- `1 ≤ keyName[i].length ≤ 10`, chỉ chữ thường.

In [1]:
import random
import time
from collections import defaultdict, deque
from typing import List

examples = [
    (["daniel", "daniel", "daniel", "luis", "luis", "luis", "luis"],
     ["10:00", "10:40", "11:00", "09:00", "11:00", "13:00", "15:00"], ["daniel"]),
    (["alice", "alice", "alice", "bob", "bob", "bob", "bob"],
     ["12:01", "12:00", "18:00", "21:00", "21:20", "21:30", "23:00"], ["bob"]),
]
for names, times, want in examples:
    print(list(zip(names, times)), "->", want)

[('daniel', '10:00'), ('daniel', '10:40'), ('daniel', '11:00'), ('luis', '09:00'), ('luis', '11:00'), ('luis', '13:00'), ('luis', '15:00')] -> ['daniel']
[('alice', '12:01'), ('alice', '12:00'), ('alice', '18:00'), ('bob', '21:00'), ('bob', '21:20'), ('bob', '21:30'), ('bob', '23:00')] -> ['bob']


## 3. Kiến thức nền

### 3.1 Đổi `"HH:MM"` sang phút

`int(s[:2]) * 60 + int(s[3:])`. Ví dụ `"10:40"` → `640`. Sau khi đổi, "trong một giờ" nghĩa là hiệu **≤ 60**.

In [2]:
def to_min(s):
    return int(s[:2]) * 60 + int(s[3:])

for s in ["00:00", "09:49", "10:40", "23:59"]:
    print(s, "->", to_min(s))
print("22:51 -> 23:52 cách", to_min("23:52") - to_min("22:51"), "phút (> 60: không tính)")
print("10:00 -> 11:00 cách", to_min("11:00") - to_min("10:00"), "phút (= 60: có tính)")

00:00 -> 0
09:49 -> 589
10:40 -> 640
23:59 -> 1439
22:51 -> 23:52 cách 61 phút (> 60: không tính)
10:00 -> 11:00 cách 60 phút (= 60: có tính)


### 3.2 Gom nhóm bằng `defaultdict(list)` và `zip`

`zip(keyName, keyTime)` duyệt song song hai danh sách.

In [3]:
groups = defaultdict(list)
for name, t in zip(examples[1][0], examples[1][1]):
    groups[name].append(to_min(t))
print(dict(groups))

{'alice': [721, 720, 1080], 'bob': [1260, 1280, 1290, 1380]}


### 3.3 Sắp xếp: `list.sort()` và `sorted()`

`lst.sort()` sắp xếp tại chỗ; `sorted(iterable)` trả list mới. Chuỗi được sắp xếp theo bảng chữ cái (thứ tự từ điển).

In [4]:
times = [721, 720, 1080]
times.sort(); print(times)
print(sorted({"luis", "bob", "daniel", "alice"}))

[720, 721, 1080]
['alice', 'bob', 'daniel', 'luis']


### 3.4 Hai con trỏ / cửa sổ trượt trên dãy đã sắp xếp

Giữ `left` sao cho `t[right] − t[left] ≤ w`; số phần tử trong cửa sổ là `right − left + 1`. Mỗi con trỏ chỉ tiến, tổng O(k).

In [5]:
t = [600, 640, 660, 700, 800]
left = 0
for right in range(len(t)):
    while t[right] - t[left] > 60:
        left += 1
    print(f"right={right} t={t[right]} cửa sổ [{left}..{right}] = {t[left:right+1]} ({right - left + 1} lần)")

right=0 t=600 cửa sổ [0..0] = [600] (1 lần)
right=1 t=640 cửa sổ [0..1] = [600, 640] (2 lần)
right=2 t=660 cửa sổ [0..2] = [600, 640, 660] (3 lần)
right=3 t=700 cửa sổ [1..3] = [640, 660, 700] (3 lần)
right=4 t=800 cửa sổ [4..4] = [800] (1 lần)


## 4. Ý tưởng nền tảng / trực giác

Xét `luis` ở ví dụ 1: `09:00, 11:00, 13:00, 15:00` → phút `540, 660, 780, 900`. Mọi cặp liền nhau cách 120 phút ⇒ không thể có 3 lần trong 60 phút.

Xét `daniel`: `600, 640, 660`. Lần đầu và lần thứ ba cách đúng 60 phút ⇒ cả 3 lần nằm trong khoảng `[10:00, 11:00]` ⇒ cảnh báo.

**Quan sát then chốt:** sau khi sắp xếp thời gian của một người, nếu có 3 lần quẹt nằm trong một khoảng 60 phút thì **3 lần liên tiếp** nào đó cũng nằm trong 60 phút (ba phần tử liên tiếp bắt đầu từ lần sớm nhất trong bộ ba sẽ "chặt" hơn). Vậy chỉ cần kiểm tra `t[i] − t[i−2] ≤ 60`.

Ví dụ 2: `alice` có `12:00, 12:01, 18:00` (lưu ý dữ liệu **không** theo thứ tự) → `18:00 − 12:00 = 360 > 60` ⇒ không cảnh báo. `bob`: `21:00, 21:20, 21:30` → `30 ≤ 60` ⇒ cảnh báo.

In [6]:
for names, times, _ in examples:
    g = defaultdict(list)
    for nm, tt in zip(names, times):
        g[nm].append(to_min(tt))
    for nm, ts in g.items():
        ts.sort()
        gaps = [ts[i] - ts[i - 2] for i in range(2, len(ts))]
        print(f"{nm:7s} phút đã sort = {ts}, t[i]-t[i-2] = {gaps}, cảnh báo = {any(x <= 60 for x in gaps)}")

daniel  phút đã sort = [600, 640, 660], t[i]-t[i-2] = [60], cảnh báo = True
luis    phút đã sort = [540, 660, 780, 900], t[i]-t[i-2] = [240, 240], cảnh báo = False
alice   phút đã sort = [720, 721, 1080], t[i]-t[i-2] = [360], cảnh báo = False
bob     phút đã sort = [1260, 1280, 1290, 1380], t[i]-t[i-2] = [30, 100], cảnh báo = True


## 5. Cách 1 — Brute Force / baseline: đếm số lần quẹt trong cửa sổ bắt đầu tại mỗi lần

**Ý tưởng:** gom theo tên (chưa cần sort). Với mỗi lần quẹt `s` của một người, đếm số lần quẹt `x` của chính người đó thỏa `s ≤ x ≤ s + 60`. Nếu có `s` nào cho số đếm ≥ 3 thì người đó bị cảnh báo. (Khoảng 60 phút "tốt nhất" luôn có thể dời để bắt đầu đúng tại một lần quẹt.)

In [7]:
def alert_brute(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_min(t))
    res = []
    for name, ts in groups.items():
        for s in ts:
            if sum(1 for x in ts if s <= x <= s + 60) >= 3:
                res.append(name)
                break
    return sorted(res)

for names, times, want in examples:
    print(alert_brute(names, times), "| mong đợi", want)

['daniel'] | mong đợi ['daniel']
['bob'] | mong đợi ['bob']


**Độ phức tạp:** với người có `k` lần quẹt, O(k²) ⇒ worst case (một người quẹt `10⁵` lần — nhưng mỗi phút chỉ một lần nên `k ≤ 1440`) là `1440² ≈ 2·10⁶` mỗi người. Tổng vẫn chấp nhận được nhờ ràng buộc "cặp (tên, giờ) duy nhất", nhưng về nguyên tắc là O(k²).

**Nút thắt:** đếm lại cửa sổ từ đầu cho mỗi điểm bắt đầu. Sắp xếp một lần giúp mọi thứ tuyến tính.

In [8]:
# Nhiều người, mỗi người quẹt cách nhau 31 phút (2 lần/giờ -> không ai bị cảnh báo, brute phải xét hết)
names, times = [], []
for p in range(2000):
    for m in range(p % 31, 1440, 31):
        names.append(f"w{p}"); times.append(f"{m // 60:02d}:{m % 60:02d}")
t0 = time.perf_counter(); r = alert_brute(names, times); print(len(names), "lần quẹt ->", len(r), "người bị cảnh báo,", f"{time.perf_counter() - t0:.3f}s")

92910 lần quẹt -> 0 người bị cảnh báo, 0.204s


## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề cửa sổ 3 phần tử:** gọi `t[0] ≤ t[1] ≤ … ≤ t[k−1]` là các lần quẹt đã sắp xếp của một người. Tồn tại 3 lần trong một khoảng dài 60 phút **khi và chỉ khi** tồn tại `i ≥ 2` với `t[i] − t[i−2] ≤ 60`.
- (⇐) hiển nhiên: `t[i−2], t[i−1], t[i]` nằm trong `[t[i−2], t[i−2] + 60]`.
- (⇒) nếu 3 lần `t[a] < t[b] < t[c]` (chỉ số `a < b < c`) nằm trong 60 phút thì `t[a+2] ≤ t[c]` (vì `a + 2 ≤ c`), nên `t[a+2] − t[a] ≤ t[c] − t[a] ≤ 60`.

**Quy tắc "nếu… thì…":**
- Nếu dãy của một người có ít hơn 3 phần tử thì chắc chắn không cảnh báo.
- Nếu có `i` mà `t[i] − t[i−2] ≤ 60` thì cảnh báo người đó (dừng kiểm tra người đó ngay).
- Kết quả: tập tên bị cảnh báo, **sắp xếp** trước khi trả.

**Invariant khi quét một người:** trước bước `i`, chưa có cặp `(j−2, j)` nào với `j < i` thỏa điều kiện.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Gom nhóm + sort từng người + cửa sổ 3 phần tử

Từng đoạn:
1. `groups[name].append(to_min(time))`.
2. Duyệt **tên theo thứ tự đã sắp xếp** (`sorted(groups)`) để kết quả tự động có thứ tự.
3. Sort thời gian của người đó; kiểm tra `any(ts[i] - ts[i-2] <= 60 ...)`.

In [9]:
def alert_group_sort(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(int(t[:2]) * 60 + int(t[3:]))
    res = []
    for name in sorted(groups):
        ts = sorted(groups[name])
        if any(ts[i] - ts[i - 2] <= 60 for i in range(2, len(ts))):
            res.append(name)
    return res

for names, times, want in examples:
    print(alert_group_sort(names, times), "| mong đợi", want)

['daniel'] | mong đợi ['daniel']
['bob'] | mong đợi ['bob']


Code nộp LeetCode của Cách 2 (xem mục 9).

**Độ phức tạp:** Time O(N log N) (tổng chi phí sort các nhóm ≤ sort toàn bộ, cộng sort tên), Space O(N).

### 7.2 Cách 3 — Sắp xếp toàn cục theo `(tên, phút)` rồi quét một lượt

Thay vì `dict`, tạo danh sách cặp `(tên, phút)` và sort một lần: các lần quẹt của cùng một người nằm **liền nhau** và đã tăng dần theo thời gian. Quét với chỉ số `i`: nếu `pairs[i−2]` cùng tên với `pairs[i]` và hiệu thời gian ≤ 60 ⇒ cảnh báo. Vì tên được sort, kết quả tự nhiên có thứ tự; chỉ cần tránh thêm trùng (so với phần tử cuối của `res`).

In [10]:
def alert_global_sort(keyName, keyTime):
    pairs = sorted((name, int(t[:2]) * 60 + int(t[3:])) for name, t in zip(keyName, keyTime))
    res = []
    for i in range(2, len(pairs)):
        name, m = pairs[i]
        if pairs[i - 2][0] == name and m - pairs[i - 2][1] <= 60:
            if not res or res[-1] != name:
                res.append(name)
    return res

for names, times, want in examples:
    print(alert_global_sort(names, times), "| mong đợi", want)

['daniel'] | mong đợi ['daniel']
['bob'] | mong đợi ['bob']


Ở đây `pairs[i−2][0] == name` kéo theo `pairs[i−1]` cũng cùng tên (vì các cặp cùng tên liền nhau).

Code nộp LeetCode của Cách 3:

```python
class Solution:
    def alertNames(self, keyName: List[str], keyTime: List[str]) -> List[str]:
        pairs = sorted((n, int(t[:2]) * 60 + int(t[3:])) for n, t in zip(keyName, keyTime))
        res = []
        for i in range(2, len(pairs)):
            name, m = pairs[i]
            if pairs[i - 2][0] == name and m - pairs[i - 2][1] <= 60:
                if not res or res[-1] != name:
                    res.append(name)
        return res
```

**Độ phức tạp:** Time O(N log N), Space O(N).

### 7.3 Cách 4 — Cửa sổ trượt tổng quát bằng `deque` (k lần trong w phút)

Viết dạng tổng quát để dễ tái sử dụng: với mỗi người (thời gian đã sort), duy trì `deque` các lần quẹt trong cửa sổ `[t − w, t]`; khi kích thước ≥ `k` ⇒ cảnh báo. Với bài gốc `k = 3`, `w = 60`.

In [11]:
def alert_window(keyName, keyTime, k=3, w=60):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(int(t[:2]) * 60 + int(t[3:]))
    res = []
    for name in sorted(groups):
        win = deque()
        for m in sorted(groups[name]):
            win.append(m)
            while m - win[0] > w:          # loại lần quẹt quá 60 phút trước
                win.popleft()
            if len(win) >= k:
                res.append(name)
                break
    return res

for names, times, want in examples:
    print(alert_window(names, times), "| mong đợi", want)

['daniel'] | mong đợi ['daniel']
['bob'] | mong đợi ['bob']


Code nộp LeetCode của Cách 4:

```python
from collections import defaultdict, deque

class Solution:
    def alertNames(self, keyName: List[str], keyTime: List[str]) -> List[str]:
        groups = defaultdict(list)
        for name, t in zip(keyName, keyTime):
            groups[name].append(int(t[:2]) * 60 + int(t[3:]))
        res = []
        for name in sorted(groups):
            win = deque()
            for m in sorted(groups[name]):
                win.append(m)
                while m - win[0] > 60:
                    win.popleft()
                if len(win) >= 3:
                    res.append(name)
                    break
        return res
```

**Độ phức tạp:** Time O(N log N) (do sort), phần cửa sổ O(N); Space O(N).

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2:

| tên | phút đã sort | i | t[i] − t[i−2] | ≤ 60? |
|---|---|---|---|---|
| alice | [720, 721, 1080] | 2 | 1080 − 720 = 360 | không |
| bob | [1260, 1280, 1290, 1380] | 2 | 1290 − 1260 = 30 | **có** ⇒ cảnh báo, dừng |

Kết quả `["bob"]`.

In [12]:
def trace_alert(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_min(t))
    print("nhóm (chưa sort):", dict(groups))
    res = []
    for name in sorted(groups):
        ts = sorted(groups[name])
        print(f"{name}: {ts}")
        for i in range(2, len(ts)):
            d = ts[i] - ts[i - 2]
            print(f"   i={i}: {ts[i]} - {ts[i-2]} = {d} -> {'CẢNH BÁO' if d <= 60 else 'không'}")
            if d <= 60:
                res.append(name); break
        if len(ts) < 3:
            print("   < 3 lần quẹt -> bỏ qua")
    print("=> kết quả", res)
    return res

trace_alert(*examples[1][:2])

nhóm (chưa sort): {'alice': [721, 720, 1080], 'bob': [1260, 1280, 1290, 1380]}
alice: [720, 721, 1080]
   i=2: 1080 - 720 = 360 -> không
bob: [1260, 1280, 1290, 1380]
   i=2: 1290 - 1260 = 30 -> CẢNH BÁO
=> kết quả ['bob']


['bob']

In [13]:
trace_alert(["a", "a", "a", "a", "b", "b"], ["23:00", "22:00", "23:59", "22:59", "01:00", "01:01"])

nhóm (chưa sort): {'a': [1380, 1320, 1439, 1379], 'b': [60, 61]}
a: [1320, 1379, 1380, 1439]
   i=2: 1380 - 1320 = 60 -> CẢNH BÁO
b: [60, 61]
   < 3 lần quẹt -> bỏ qua
=> kết quả ['a']


['a']

Trong trace thứ hai: `a` có `[1320, 1379, 1380, 1439]`; `i = 2`: `1380 − 1320 = 60 ≤ 60` ⇒ cảnh báo ngay. `b` chỉ có 2 lần ⇒ bỏ qua.

## 9. Code nộp LeetCode

```python
from collections import defaultdict
from typing import List

class Solution:
    def alertNames(self, keyName: List[str], keyTime: List[str]) -> List[str]:
        groups = defaultdict(list)
        for name, t in zip(keyName, keyTime):
            groups[name].append(int(t[:2]) * 60 + int(t[3:]))   # "HH:MM" -> phút
        res = []
        for name in sorted(groups):                              # tên theo thứ tự
            ts = sorted(groups[name])
            if any(ts[i] - ts[i - 2] <= 60 for i in range(2, len(ts))):
                res.append(name)
        return res
```

In [14]:
class Solution:
    def alertNames(self, keyName: List[str], keyTime: List[str]) -> List[str]:
        groups = defaultdict(list)
        for name, t in zip(keyName, keyTime):
            groups[name].append(int(t[:2]) * 60 + int(t[3:]))
        res = []
        for name in sorted(groups):
            ts = sorted(groups[name])
            if any(ts[i] - ts[i - 2] <= 60 for i in range(2, len(ts))):
                res.append(name)
        return res

for names, times, want in examples:
    got = Solution().alertNames(names, times)
    print(got, "OK" if got == want else "SAI")

['daniel'] OK
['bob'] OK


## 10. Vì sao thuật toán đúng

1. **Đổi đơn vị đúng:** mọi thời điểm cùng một ngày, nên số phút từ 0:00 là một ánh xạ bảo toàn thứ tự và khoảng cách; "trong một giờ" ⇔ hiệu ≤ 60 (theo ví dụ biên của đề: 60 phút là tính, 61 phút là không).
2. **Độc lập giữa các người:** cảnh báo chỉ xét các lần quẹt của **cùng** một người ⇒ gom nhóm theo tên.
3. **Bổ đề cửa sổ 3 phần tử** (mục 6) cho thấy điều kiện `∃ i: t[i] − t[i−2] ≤ 60` tương đương với "có 3 lần trong một giờ".
4. **Định dạng kết quả:** duyệt tên theo `sorted(groups)` và thêm mỗi tên tối đa một lần ⇒ danh sách không trùng, tăng dần.

## 11. Phân tích Time / Space

Gọi `N = len(keyName)`, `k_p` = số lần quẹt của người `p` (`Σ k_p = N`, `k_p ≤ 1440`), `P` = số người.

| Cách | Time | Space |
|---|---|---|
| 1. Brute | O(Σ k_p²) ≤ O(N · 1440) | O(N) |
| 2. Gom + sort + cửa sổ 3 | O(N log N + P log P) | O(N) |
| 3. Sort toàn cục | O(N log N) | O(N) |
| 4. Deque tổng quát | O(N log N) | O(N) |

(Thực tế có thể sort thời gian bằng **counting sort** 1440 ô cho mỗi người nhiều lần quẹt, nhưng `sorted` của Python đủ nhanh.) Đo với `N = 10⁵`:

In [15]:
random.seed(0)
N = 10**5
pool = set()
while len(pool) < N:
    pool.add((f"u{random.randint(0, 3000)}", random.randint(0, 1439)))
pool = list(pool); random.shuffle(pool)
big_names = [p for p, _ in pool]
big_times = [f"{m // 60:02d}:{m % 60:02d}" for _, m in pool]
for f in [alert_group_sort, alert_global_sort, alert_window]:
    t0 = time.perf_counter(); r = f(big_names, big_times)
    print(f"{f.__name__:18s} -> {len(r)} người bị cảnh báo, {time.perf_counter() - t0:.3f}s")

alert_group_sort   -> 3000 người bị cảnh báo, 0.081s
alert_global_sort  -> 3000 người bị cảnh báo, 0.192s


alert_window       -> 3000 người bị cảnh báo, 0.050s


## 12. Bộ test

Ví dụ đề, biên 60/61 phút, dữ liệu không sắp xếp, người có < 3 lần, nhiều người bị cảnh báo (kiểm tra thứ tự kết quả), quanh nửa đêm (`00:00`, `23:59` — cùng ngày, không vòng qua đêm), và đối chiếu ngẫu nhiên (cặp tên–giờ duy nhất như đề).

In [16]:
ALL = [alert_brute, alert_group_sort, alert_global_sort, alert_window,
       lambda a, b: Solution().alertNames(a, b)]
cases = [(n, t, w) for n, t, w in examples] + [
    (["a", "a", "a"], ["10:00", "10:30", "11:00"], ["a"]),           # đúng 60 phút
    (["a", "a", "a"], ["10:00", "10:30", "11:01"], []),              # 61 phút
    (["a", "a", "a"], ["22:51", "23:00", "23:52"], []),              # ví dụ biên của đề
    (["a", "a"], ["10:00", "10:01"], []),                            # < 3 lần
    (["z", "z", "z", "b", "b", "b"], ["01:00", "01:01", "01:02", "05:00", "05:10", "05:20"], ["b", "z"]),
    (["x", "x", "x"], ["23:59", "00:00", "00:01"], []),              # không vòng qua nửa đêm
    (["a", "a", "a", "a"], ["12:00", "11:00", "09:00", "13:00"], []),
    (["a", "b", "a", "b", "a", "b"], ["10:00", "10:00", "10:20", "10:20", "10:40", "12:00"], ["a"]),
]
for n, t, w in cases:
    for f in ALL:
        assert f(n, t) == w, (f, n, t, f(n, t), w)
print("Ca cố định:", len(cases), "OK")

rnd = random.Random(2024)
for _ in range(1500):
    P = rnd.randint(1, 4); M = rnd.randint(1, 15)
    span = rnd.choice([90, 240, 1440])
    base = rnd.randint(0, 1440 - span)
    seen = set()
    for _ in range(M):
        seen.add((rnd.choice("abcd"[:P]), base + rnd.randrange(span)))
    lst = list(seen); rnd.shuffle(lst)
    n = [p for p, _ in lst]; t = [f"{m // 60:02d}:{m % 60:02d}" for _, m in lst]
    want = alert_brute(n, t)
    for f in ALL[1:]:
        assert f(n, t) == want, (f, n, t)
print("Ngẫu nhiên: OK")

Ca cố định: 10 OK
Ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `HHMM` dạng số nguyên với ngưỡng 60

In [17]:
def wrong_hhmm(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(int(t.replace(":", "")))     # SAI: "10:50" -> 1050
    res = []
    for name in sorted(groups):
        ts = sorted(groups[name])
        if any(ts[i] - ts[i - 2] <= 60 for i in range(2, len(ts))):
            res.append(name)
    return res

n, t = ["a", "a", "a"], ["10:50", "11:00", "11:20"]
print("sai :", wrong_hhmm(n, t), "| đúng:", Solution().alertNames(n, t))

sai : [] | đúng: ['a']


**Vì sao sai:** `11:20` dạng số là `1120`, `10:50` là `1050`; hiệu `70 > 60` dù thực tế chỉ cách 30 phút. Giờ có 60 phút chứ không phải 100. **Sửa:** đổi sang phút: `h * 60 + m`.

### Lỗi 2 — Dùng `< 60` thay vì `<= 60`

In [18]:
def wrong_strict(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_min(t))
    return [name for name in sorted(groups)
            if any(sorted(groups[name])[i] - sorted(groups[name])[i - 2] < 60     # SAI: '<'
                   for i in range(2, len(groups[name])))]

n, t, _ = examples[0]
print("sai :", wrong_strict(n, t), "| đúng:", Solution().alertNames(n, t))

sai : [] | đúng: ['daniel']


**Vì sao sai:** `daniel` quẹt `10:00, 10:40, 11:00` — hiệu đúng 60 phút, đề nói rõ `10:00`–`11:00` là trong một giờ. **Sửa:** `<= 60`.

### Lỗi 3 — Quên sắp xếp thời gian

In [19]:
def wrong_no_sort(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_min(t))
    res = []
    for name in sorted(groups):
        ts = groups[name]                                  # SAI: giữ nguyên thứ tự đầu vào
        if any(ts[i] - ts[i - 2] <= 60 for i in range(2, len(ts))):
            res.append(name)
    return res

n, t = ["a", "a", "a"], ["12:00", "11:00", "09:00"]
print("sai :", wrong_no_sort(n, t), "| đúng:", Solution().alertNames(n, t))

sai : ['a'] | đúng: []


**Vì sao sai:** dữ liệu **không** theo thứ tự thời gian (ví dụ 2 của đề có `"12:01"` trước `"12:00"`). Không sort thì `t[i] − t[i−2]` có thể **âm** (`540 − 720 = −180 ≤ 60`) ⇒ cảnh báo nhầm. **Sửa:** sort thời gian của từng người.

### Lỗi 4 — Thêm tên trùng lặp / không sắp xếp kết quả

In [20]:
def wrong_dup(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_min(t))
    res = []
    for name, ts in groups.items():                       # SAI: thứ tự theo lần xuất hiện
        ts.sort()
        for i in range(2, len(ts)):
            if ts[i] - ts[i - 2] <= 60:
                res.append(name)                          # SAI: không break -> trùng tên
    return res

n = ["zed", "zed", "zed", "zed", "amy", "amy", "amy"]
t = ["08:00", "08:10", "08:20", "08:30", "09:00", "09:01", "09:02"]
print("sai :", wrong_dup(n, t), "| đúng:", Solution().alertNames(n, t))

sai : ['zed', 'zed', 'amy'] | đúng: ['amy', 'zed']


**Vì sao sai:** đề yêu cầu tên **không trùng** và **sắp xếp theo chữ cái**. `zed` có nhiều bộ ba thỏa nên bị thêm nhiều lần, và `zed` đứng trước `amy`. **Sửa:** `break` sau khi thêm (hoặc dùng `set`), rồi trả `sorted(...)`.

### Lỗi 5 — Không gom theo tên (trộn lần quẹt của nhiều người)

In [21]:
def wrong_no_group(keyName, keyTime):
    pairs = sorted((to_min(t), name) for name, t in zip(keyName, keyTime))   # SAI: sort theo thời gian trước
    res = set()
    for i in range(2, len(pairs)):
        if pairs[i][0] - pairs[i - 2][0] <= 60:
            res.add(pairs[i][1])                         # gán cảnh báo cho người quẹt thứ 3
    return sorted(res)

n, t = ["a", "b", "c"], ["10:00", "10:10", "10:20"]
print("sai :", wrong_no_group(n, t), "| đúng:", Solution().alertNames(n, t))

sai : ['c'] | đúng: []


**Vì sao sai:** ba người khác nhau mỗi người quẹt một lần — không ai quẹt 3 lần. Cảnh báo phải tính **riêng từng người**. **Sửa:** gom theo tên (Cách 2) hoặc sort theo `(tên, phút)` và kiểm tra cùng tên (Cách 3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu thời gian dạng chuỗi ⇒ **chuẩn hóa sang số nhỏ nhất có nghĩa** (phút/giây) ngay từ đầu.
- "Ai đó làm X ít nhất `k` lần trong khoảng `w`" ⇒ **gom theo người → sort → cửa sổ**. Với `k` cố định nhỏ: kiểm tra `t[i] − t[i−k+1] ≤ w`.
- Đọc kỹ biên: "≤" hay "<" — đề thường cho ví dụ biên (ở đây `10:00`–`11:00`).
- Kết quả "unique + sorted" ⇒ duyệt khóa theo `sorted(dict)` và `break` sau khi thêm, hoặc `sorted(set)`.
- Đầu vào **không đảm bảo thứ tự** trừ khi đề nói rõ.

## 15. Biến thể của bài

1. **`k` lần trong `w` phút** (bài tập 4): `t[i] − t[i−k+1] ≤ w`.
2. **Thời gian vòng qua nửa đêm / nhiều ngày:** cần thêm ngày hoặc nhân đôi dãy (`t + 1440`).
3. **Trả thời điểm cảnh báo đầu tiên** của mỗi người (bài tập 5).
4. **LC 1094 — Car Pooling**, **LC 253 — Meeting Rooms II**: đếm sự kiện chồng lấn trong khoảng thời gian.
5. **LC 1152 — Analyze User Website Visit Pattern**: gom theo người, sort theo thời gian, rồi xử lý chuỗi.
6. **Rate limiting** (hệ thống thật): cửa sổ trượt bằng `deque` — chính là Cách 4.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | mỗi điểm bắt đầu, đếm trong 60 phút | O(Σk²) | O(N) | baseline, kiểm chứng |
| 2. Gom + sort + cửa sổ 3 | `t[i] − t[i−2] ≤ 60` | O(N log N) | O(N) | **nộp bài** — ngắn gọn |
| 3. Sort toàn cục | sort `(tên, phút)` một lần, quét | O(N log N) | O(N) | không cần `dict` |
| 4. Deque tổng quát | cửa sổ trượt `k` lần trong `w` phút | O(N log N) | O(N) | khi `k`, `w` thay đổi |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `keyName = ["john","john","john"]`, `keyTime = ["23:58","23:59","00:01"]`? Và `["leslie"]*5` với `["13:00","13:20","14:00","18:00","18:51"]`?

**Bài 2 (tính tay):** Đổi sang phút và tính các hiệu `t[i] − t[i−2]` cho `["08:59","09:30","10:00","09:58"]`; người này có bị cảnh báo không?

**Bài 3 (điền TODO):** Hoàn thiện `to_minutes(s)` (không dùng `split`) và `from_minutes(m)` trả về `"HH:MM"`.

**Bài 4 (tự viết lại):** Viết `alert_k_w(keyName, keyTime, k, w)` — cảnh báo nếu quẹt ≥ `k` lần trong một khoảng `w` phút (bao gồm biên).

**Bài 5 (mini challenge):** Viết `first_alert_time(keyName, keyTime)` trả `dict` tên → thời điểm `"HH:MM"` **sớm nhất** mà hệ thống có thể phát cảnh báo (thời điểm lần quẹt thứ 3 của bộ ba sớm nhất thỏa điều kiện).

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def to_minutes(s):
    # TODO: lấy 2 ký tự đầu làm giờ, 2 ký tự cuối làm phút
    pass

def from_minutes(m):
    # TODO: dùng divmod(m, 60) và định dạng 2 chữ số f"{h:02d}:{mm:02d}"
    pass

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def alert_k_w(keyName, keyTime, k, w):
    # TODO
    pass

def first_alert_time(keyName, keyTime):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: thời gian cùng một ngày, không vòng qua đêm. Mức 2: `john`: `1, 1438, 1439` sau khi sort. Mức 3: `john` → `[]`; `leslie`: `780, 800, 840, 1080, 1131` ⇒ `840 − 780 = 60` ⇒ `["leslie"]`.

**Bài 2.** Mức 1: phút `539, 570, 600, 598`. Mức 2: sort `539, 570, 598, 600`. Mức 3: hiệu `59, 30` ⇒ có cảnh báo.

**Bài 3.** Mức 1: `s[:2]`, `s[3:]`. Mức 2: `int(...)`. Mức 3: `from_minutes`: `h, mm = divmod(m, 60)`.

**Bài 4.** Mức 1: tổng quát hóa chỉ số `i − 2` thành `i − k + 1`. Mức 2: cẩn thận `k = 1` (mọi người có ít nhất 1 lần quẹt đều bị cảnh báo). Mức 3: đối chiếu với brute force đếm trong cửa sổ.

**Bài 5.** Mức 1: thời điểm sớm nhất là `t[i]` nhỏ nhất thỏa `t[i] − t[i−2] ≤ 60`. Mức 2: duyệt `i` tăng dần, dừng ở lần đầu. Mức 3: đổi lại sang `"HH:MM"`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
b1a = Solution().alertNames(["john"] * 3, ["23:58", "23:59", "00:01"])
b1b = Solution().alertNames(["leslie"] * 5, ["13:00", "13:20", "14:00", "18:00", "18:51"])
print("Bài 1:", b1a, b1b)
assert b1a == [] and b1b == ["leslie"]
assert trace_alert(["p"] * 4, ["08:59", "09:30", "10:00", "09:58"]) == ["p"]

Bài 1: [] ['leslie']
nhóm (chưa sort): {'p': [539, 570, 600, 598]}
p: [539, 570, 598, 600]
   i=2: 598 - 539 = 59 -> CẢNH BÁO
=> kết quả ['p']


In [25]:
# Bài 3
def to_minutes_sol(s):
    return int(s[:2]) * 60 + int(s[3:])

def from_minutes_sol(m):
    h, mm = divmod(m, 60)
    return f"{h:02d}:{mm:02d}"

for m in range(1440):
    assert to_minutes_sol(from_minutes_sol(m)) == m
assert from_minutes_sol(0) == "00:00" and from_minutes_sol(1439) == "23:59"
print("Bài 3: OK")

Bài 3: OK


In [26]:
# Bài 4
def alert_k_w_sol(keyName, keyTime, k, w):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_minutes_sol(t))
    res = []
    for name in sorted(groups):
        ts = sorted(groups[name])
        if any(ts[i] - ts[i - k + 1] <= w for i in range(k - 1, len(ts))):
            res.append(name)
    return res

def alert_k_w_brute(keyName, keyTime, k, w):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_minutes_sol(t))
    return sorted(n for n, ts in groups.items() if any(sum(s <= x <= s + w for x in ts) >= k for s in ts))

rnd = random.Random(4)
for _ in range(800):
    seen = set()
    for _ in range(rnd.randint(1, 14)):
        seen.add((rnd.choice("abc"), rnd.randrange(300)))
    n = [p for p, _ in seen]; t = [from_minutes_sol(m) for _, m in seen]
    k, w = rnd.randint(1, 5), rnd.randint(0, 120)
    assert alert_k_w_sol(n, t, k, w) == alert_k_w_brute(n, t, k, w)
for n, t, want in examples:
    assert alert_k_w_sol(n, t, 3, 60) == want
print("Bài 4: OK")

Bài 4: OK


In [27]:
# Bài 5
def first_alert_time_sol(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_minutes_sol(t))
    res = {}
    for name in sorted(groups):
        ts = sorted(groups[name])
        for i in range(2, len(ts)):
            if ts[i] - ts[i - 2] <= 60:
                res[name] = from_minutes_sol(ts[i]); break
    return res

def first_alert_brute(keyName, keyTime):
    groups = defaultdict(list)
    for name, t in zip(keyName, keyTime):
        groups[name].append(to_minutes_sol(t))
    res = {}
    for name, ts in groups.items():
        best = None
        for c in ts:                        # c là thời điểm lần quẹt thứ 3 của một bộ ba
            if sum(1 for x in ts if c - 60 <= x <= c) >= 3:
                best = c if best is None else min(best, c)
        if best is not None:
            res[name] = from_minutes_sol(best)
    return res

print("Bài 5 ví dụ 2:", first_alert_time_sol(*examples[1][:2]))
rnd = random.Random(5)
for _ in range(800):
    seen = set()
    for _ in range(rnd.randint(1, 14)):
        seen.add((rnd.choice("abc"), rnd.randrange(300)))
    n = [p for p, _ in seen]; t = [from_minutes_sol(m) for _, m in seen]
    assert first_alert_time_sol(n, t) == first_alert_brute(n, t)
print("Bài 5: OK")

Bài 5 ví dụ 2: {'bob': '21:30'}


Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "≥ k sự kiện trong w đơn vị, theo từng người":**

```python
groups = defaultdict(list)
for name, t in zip(keyName, keyTime):
    groups[name].append(int(t[:2]) * 60 + int(t[3:]))
res = []
for name in sorted(groups):
    ts = sorted(groups[name])
    if any(ts[i] - ts[i - k + 1] <= w for i in range(k - 1, len(ts))):
        res.append(name)
return res
```

**Checklist:**
- [ ] Đổi `"HH:MM"` → phút bằng `h * 60 + m` (không dùng `HHMM` nguyên)?
- [ ] Sort thời gian của từng người?
- [ ] Biên `<= 60` (bao gồm)?
- [ ] Tính riêng từng người (gom theo tên)?
- [ ] Kết quả không trùng, sắp xếp theo tên?
- [ ] Đã test: đúng 60 / 61 phút, dữ liệu lộn xộn, `< 3` lần, nhiều người bị cảnh báo, gần nửa đêm?